In [1]:
!pip install -q transformers accelerate qwen-vl-utils datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 43.3 MB/s eta 0:00:00


In [3]:
import torch
import os
import time
import json
from tqdm import tqdm
from google.colab import userdata
from datasets import load_dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from huggingface_hub import snapshot_download

# --- A. LẤY TOKEN VÀ LOAD DATASET ---
# Lấy token từ phần Secrets của Colab
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None
    print("⚠️ Lưu ý: Chưa thiết lập HF_TOKEN trong phần Secrets của Colab.")

print("📂 Đang tải tập dữ liệu VALIDATION...")

# Load duy nhất tập test bằng cách chỉ định data_files
# Cách này giúp Colab không tải toàn bộ 8000 ảnh mà chỉ tải những ảnh trong thư mục test
dataset = load_dataset(
    "pqthinh232/HCMUS-Vietnamese-Image-captioning-for-visually-impaired",
    data_files={"validation": "val/**"},
    split="validation",
    token=hf_token
)

print(f"✅ Thành công! Đã nạp {len(dataset)} ảnh tập val vào bộ nhớ.")

📂 Đang tải tập dữ liệu VALIDATION...


Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

✅ Thành công! Đã nạp 4000 ảnh tập val vào bộ nhớ.


In [4]:
# --- B. KHỞI TẠO MODEL QWEN2-VL ---
MODEL_ID = "pqthinh232/HCMUS-Qwen2-VL-2B-Instruct-Vietnamese-Image-Captioning-for-blind-E1"
OUTPUT_FILE = "results_qwen2vl_val_E1.jsonl"

print("🚀 Đang khởi tạo model Qwen2-VL-2B...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

def get_actual_disk_size(model_id):
    # Lấy đường dẫn thư mục snapshot của model
    model_path = snapshot_download(repo_id=model_id, local_files_only=True)
    total_size = 0

    for dirpath, dirnames, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            real_fp = os.path.realpath(fp)
            total_size += os.path.getsize(real_fp)

    return total_size / (1024**3) # Trả về đơn vị GB

# Tính Params và Disk size
total_params = sum(p.numel() for p in model.parameters())
disk_size_gb = get_actual_disk_size(MODEL_ID)

🚀 Đang khởi tạo model Qwen2-VL-2B...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [5]:
# --- C. CHẠY ZERO-SHOT INFERENCE ---
prompt_text = (
    "Viết đúng một câu ngắn (tối đa 60 từ) bằng tiếng Việt, mô tả vật thể hoặc chướng ngại chính trong ảnh và đưa ra hướng dẫn di chuyển an toàn cho người mù, không giải thích thêm."
)

# 1. Lọc lấy danh sách ảnh duy nhất dựa trên tên file thật
unique_images = {}
print("🔍 Đang trích xuất tên file gốc và lọc 800 ảnh duy nhất...")

for item in dataset:
    # Lấy đường dẫn đầy đủ từ đối tượng image
    # Hugging Face lưu đường dẫn gốc tại thuộc tính .filename
    full_path = item['image'].filename

    if full_path:
        # Lấy phần tên file cuối cùng (ví dụ: '00009.jpg')
        f_name = os.path.basename(full_path)
    else:
        # Nếu không lấy được filename (rất hiếm), dùng index làm định danh
        f_name = f"unknown_{hash(item['caption'])}.jpg"

    if f_name not in unique_images:
        unique_images[f_name] = item['image']

unique_file_names = sorted(list(unique_images.keys()))
print(f"✅ Thành công! Đã tìm thấy {len(unique_file_names)} ảnh với tên gốc chuẩn.")

# 2. Bắt đầu Inference
total_time = 0
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"🚀 Bắt đầu xử lý {len(unique_file_names)} ảnh...")

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for f_name in tqdm(unique_file_names):
        image = unique_images[f_name]

        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt_text}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, padding=True, return_tensors="pt").to("cuda")

        t0 = time.time()
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=60,
                temperature=0.2,
                top_p=0.9,
                repetition_penalty=1.1,
                do_sample=False,
                eos_token_id=processor.tokenizer.eos_token_id
            )
        t1 = time.time()
        total_time += (t1 - t0)

        output_text = processor.batch_decode(generated_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

        # Làm sạch: xóa xuống dòng và các nhãn thừa
        final_caption = output_text.replace('\n', ' ').strip()
        for junk in ["Mô tả:", "Lời khuyên:", "Trả lời:"]:
            final_caption = final_caption.replace(junk, "")

        res_item = {
            "file_name": f_name, # Bây giờ sẽ là '00009.jpg' chứ không phải 'img_00000.jpg'
            "prediction": final_caption.strip()
        }
        f.write(json.dumps(res_item, ensure_ascii=False) + "\n")

# --- D. XUẤT BÁO CÁO HIỆU NĂNG ---
avg_time = total_time / len(unique_file_names)
peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

🔍 Đang trích xuất tên file gốc và lọc 800 ảnh duy nhất...
✅ Thành công! Đã tìm thấy 800 ảnh với tên gốc chuẩn.
🚀 Bắt đầu xử lý 800 ảnh...


100%|██████████| 800/800 [34:48<00:00,  2.61s/it]


In [6]:
print("\n" + "="*40)
print("📊 EFFICIENCY METRICS")
print(f"- Params: {total_params / 1e9:.2f} B")
print(f"- Disk Size: {disk_size_gb:.2f} GB")
print(f"- Time/Img: {avg_time:.4f} s")
print(f"- Peak VRAM: {peak_vram:.2f} GB")
print(f"- Results have been saved at: {OUTPUT_FILE}")
print("="*40)


📊 EFFICIENCY METRICS
- Params: 2.21 B
- Disk Size: 4.13 GB
- Time/Img: 2.5811 s
- Peak VRAM: 4.23 GB
- Results have been saved at: results_qwen2vl_val_E1.jsonl
